In [1]:


!pip install -q ftfy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.4 MB/s eta 0:00:00


In [2]:


import ftfy
import pandas as pd

INPUT_PATH = "/kaggle/input/datasets/danielantoniudumitru/job-application-spanish/job_applicant_dataset_es.csv"
OUTPUT_PATH = "/kaggle/working/job_applicant_dataset_es_fixed.csv"

COLS_TO_FIX = ["Resume", "Job Description", "Job Roles"]

In [3]:


df = pd.read_csv(INPUT_PATH, encoding="latin-1")
print(f"Shape: {df.shape}")
print("\nBefore fix — sample Resume row 0:")
print(df["Resume"].iloc[0][:300])

Shape: (10000, 9)

Before fix — sample Resume row 0:
Adquirir informaciÃ³n sobre la salud y la salud en el lugar de trabajo.Daisuke Mori 243 Hill Street Amsterdam, North Holland +31 20 018-1590 daisuke_mori@protonmail.com https://www.linkedin.com/in/daisukemori --- RESUMEN PROFESSIONAL Fitness Coach con 2-4 aÃ±os de experiencia prÃ¡ctica en PrevenciÃ³


In [4]:


def fix_mojibake(text):
    if not text or str(text) == "nan":
        return text

    text = str(text)
    try:
        fixed = text.encode("latin-1").decode("utf-8")
        return ftfy.fix_text(fixed)
    except (UnicodeEncodeError, UnicodeDecodeError):
        return ftfy.fix_text(text)


for col in COLS_TO_FIX:
    df[col] = df[col].apply(
        lambda x: fix_mojibake(x) if pd.notna(x) else x
    )
    print(f"  Fixed: {col}")

print("\nAfter fix — sample Resume row 0:")
print(df["Resume"].iloc[0][:300])

  Fixed: Resume
  Fixed: Job Description
  Fixed: Job Roles

After fix — sample Resume row 0:
Adquirir información sobre la salud y la salud en el lugar de trabajo.Daisuke Mori 243 Hill Street Amsterdam, North Holland +31 20 018-1590 daisuke_mori@protonmail.com https://www.linkedin.com/in/daisukemori --- RESUMEN PROFESSIONAL Fitness Coach con 2-4 años de experiencia práctica en Prevención de


In [5]:


df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"\nSaved to: {OUTPUT_PATH}")


Saved to: /kaggle/working/job_applicant_dataset_es_fixed.csv


In [6]:


problem_chars = ["Ã³", "Ã©", "Ã±", "Ã¡", "Ã¼", "Ã", "â€"]
print("\nChecking for remaining mojibake …")
for col in COLS_TO_FIX:
    for char in problem_chars:
        count = df[col].str.contains(char, regex=False, na=False).sum()
        if count > 0:
            print(f"  WARNING: '{char}' still found {count} times in {col}")
print("Done. No output = all clean.")


Checking for remaining mojibake …
Done. No output = all clean.
